# **1. Model Loading**

In [29]:
# Install libraries
!pip install transformers torch accelerate --quiet

# Import required libraries
import os
import warnings
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging

# Disable warnings and progress bars
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"
logging.set_verbosity_error()
warnings.filterwarnings('ignore')


# Load Qwen 2.5 model (excellent for conversations and Q&A)
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model with automatic device placement
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()
print("Model loaded successfully!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


# **2. Response Generation**

In [30]:
def generate_reply(chat_history, user_message, max_new_tokens=256):
    # Append user message to history
    chat_history.append({"role": "user", "content": user_message})

    # Apply Qwen chat template
    prompt_text = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Tokenize the prompt
    inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)

    # Generate response
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05
        )

    # Extract only the newly generated tokens
    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]

    # Decode the assistant's reply
    assistant_reply = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # Append assistant reply to history
    chat_history.append({"role": "assistant", "content": assistant_reply})

    return chat_history, assistant_reply

# **3. Continuous Conversation and Exit loop**

In [31]:
def run_chatbot():
    # Welcome message
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

    # Initialize chat history with system message
    chat_history = [
        {
            "role": "system",
            "content": "You are a helpful, concise AI assistant. Answer clearly and politely."
        }
    ]

    while True:
        # Get user input
        user_input = input("User: ").strip()

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a great day.")
            break

        # Skip empty input
        if not user_input:
            print("Chatbot: Please type something so I can respond.")
            continue

        # Generate reply
        chat_history, assistant_reply = generate_reply(chat_history, user_input)

        # Fallback if response is empty
        if assistant_reply == "":
            assistant_reply = "I'm not sure how to respond to that yet, but I'm learning."

        # Display response
        print("Chatbot:", assistant_reply)

# **Run Chatbot**

In [32]:
# Start the chatbot
run_chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?
User: hello
Chatbot: Hello! How can I help you today?
User: what is artificial intelligence?
Chatbot: Artificial Intelligence (AI) is a branch of computer science that focuses on creating intelligent machines capable of performing tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation. AI systems use algorithms and statistical models to process information and learn from data, allowing them to improve their performance over time without explicit programming. This technology has numerous applications in various fields, including healthcare, transportation, entertainment, and more.
User: Who created Python?
Chatbot: Python was created by Guido van Rossum in the late 1980s while he was a researcher at the Centrum Wiskunde & Informatica (CWI) in the Netherlands. He started developing it in 1989 and released the first version in 1991. Guido van